#  Voice2Vision — GPU Web App (Free on Colab T4)

Convert spoken voice into **Images** and **Animated Videos** using authentic AI models — running **100% Free** on Google Colab's T4 GPU.

### Models Used:
1. **Speech-to-Text**: `OpenMOSS-Team/MOSS-Transcribe-Diarize` (0.9B CausalLM)
2. **Image Generation**: `dreamlike-art/dreamlike-diffusion-1.0` (Original Stable Diffusion, **no watermarks**)
3. **Video Generation**: `AnimateDiffPipeline` with `AnimateLCM` + `emilianJR/epiCRealism` (Real 16-frame AI Video)

---
**Setup Check:** Make sure you selected **Runtime > Change runtime type > T4 GPU** before running.

## Step 1: Install Dependencies
Click the **Play (▶)** button on the cell below to download and install MOSS and AI libraries.

In [1]:
# 1. Verify GPU
!nvidia-smi

# 2. Clone MOSS-Transcribe-Diarize repository
import os
if not os.path.exists('/content/MOSS-Transcribe-Diarize'):
    print("Cloning MOSS-Transcribe-Diarize...")
    !git clone https://github.com/OpenMOSS/MOSS-Transcribe-Diarize.git /content/MOSS-Transcribe-Diarize

# 3. Install MOSS and required packages
!pip install -q -e /content/MOSS-Transcribe-Diarize
!pip install -q diffusers transformers accelerate peft "torchao>=0.16.0" gradio

print("\n✅ Step 1 complete! Now click the Play button on Step 2 below.")

Mon Sep 14 10:58:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

##  Step 2: Load Models & Launch Web App
Click the **Play (▶)** button on this cell. It will load the models onto the T4 GPU and give you your **free public link** (`https://xxxx.gradio.live`) to open the web app!

In [ ]:
import sys, os, importlib

# Ensure MOSS directory is in sys.path and caches invalidated
REPO_DIR = '/content/MOSS-Transcribe-Diarize'
if os.path.exists(REPO_DIR) and REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

importlib.invalidate_caches()

# Auto-heal import if needed
try:
    from moss_transcribe_diarize import parse_transcript
    from moss_transcribe_diarize.inference_utils import (
        build_transcription_messages,
        generate_transcription,
    )
except ModuleNotFoundError:
    print("Reloading path caches for MOSS...")
    if not os.path.exists(REPO_DIR):
        !git clone https://github.com/OpenMOSS/MOSS-Transcribe-Diarize.git /content/MOSS-Transcribe-Diarize
        !pip install -q -e /content/MOSS-Transcribe-Diarize
    sys.path.insert(0, REPO_DIR)
    importlib.invalidate_caches()
    from moss_transcribe_diarize import parse_transcript
    from moss_transcribe_diarize.inference_utils import (
        build_transcription_messages,
        generate_transcription,
    )

import torch
from transformers import AutoModelForCausalLM, AutoProcessor
from diffusers import StableDiffusionPipeline, AnimateDiffPipeline, LCMScheduler, MotionAdapter
from diffusers.utils import export_to_gif

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
moss_dtype = torch.bfloat16 if device.type == 'cuda' else torch.float32
pipe_dtype = torch.float16 if device.type == 'cuda' else torch.float32

print(f"Runtime Device: {device}")

# 1. Load MOSS Speech Model
print("\n[1/3] Loading MOSS-Transcribe-Diarize Speech Model...")
moss_model = AutoModelForCausalLM.from_pretrained(
    'OpenMOSS-Team/MOSS-Transcribe-Diarize',
    trust_remote_code=True,
    dtype='auto',
    attn_implementation='sdpa' if device.type == 'cuda' else None,
).to(dtype=moss_dtype, device=device).eval()

moss_processor = AutoProcessor.from_pretrained(
    'OpenMOSS-Team/MOSS-Transcribe-Diarize',
    trust_remote_code=True,
)

# 2. Load Dreamlike Diffusion (Images)
print("\n[2/3] Loading Dreamlike Diffusion 1.0 (No watermark)...")
image_pipe = StableDiffusionPipeline.from_pretrained(
    'dreamlike-art/dreamlike-diffusion-1.0',
    torch_dtype=pipe_dtype,
    use_safetensors=True,
).to(device)

# 3. Load AnimateDiff + AnimateLCM (Videos)
print("\n[3/3] Loading AnimateDiff + AnimateLCM (Real AI Video)...")
adapter = MotionAdapter.from_pretrained(
    'wangfuyun/AnimateLCM',
    torch_dtype=pipe_dtype,
)
video_pipe = AnimateDiffPipeline.from_pretrained(
    'emilianJR/epiCRealism',
    motion_adapter=adapter,
    torch_dtype=pipe_dtype,
)
video_pipe.scheduler = LCMScheduler.from_config(
    video_pipe.scheduler.config,
    beta_schedule='linear',
)
video_pipe.load_lora_weights(
    'wangfuyun/AnimateLCM',
    weight_name='AnimateLCM_sd15_t2v_lora.safetensors',
    adapter_name='lcm-lora',
)
video_pipe.set_adapters(['lcm-lora'], [0.8])
video_pipe.enable_model_cpu_offload()

print("\n✅ All models loaded into GPU!")

# ============================================================
# 3. Launch Gradio Web App
# ============================================================
import gradio as gr

NEGATIVE_PROMPT = (
    "bad pose, awkward pose, unnatural posture, broken posture, "
    "broken fingers, malformed fingers, extra fingers, missing fingers, "
    "deformed hands, deformed feet, deformed legs, deformed arms, "
    "long limbs, short limbs, twisted limbs, unnatural joints, "
    "facial asymmetry, distorted mouth, distorted teeth, distorted nose, "
    "distorted ears, deformed eyes, uneven eyes, blinking artifacts, "
    "face melting, face warping, face morphing, identity drift, "
    "skin flickering, hair flickering, hair changing, "
    "clothing deformation, clothing flickering, clothing morphing, "
    "accessory deformation, jewelry deformation"
)

def transcribe_voice(audio_file):
    if not audio_file:
        return "❌ Please record or upload an audio file."
    try:
        messages = build_transcription_messages(audio_file)
        result = generate_transcription(
            moss_model,
            moss_processor,
            messages,
            max_new_tokens=2048,
            do_sample=False,
            device=device,
            dtype=moss_dtype,
        )
        segments = parse_transcript(result["text"])
        transcript = " ".join(
            seg.text.strip() for seg in segments if seg.text.strip()
        )
        return transcript if transcript else "(No speech detected)"
    except Exception as e:
        return f"Error during transcription: {str(e)}"

def generate_image_fn(prompt):
    if not prompt or prompt.startswith("❌"):
        return None
    try:
        image = image_pipe(prompt).images[0]
        torch.cuda.empty_cache()
        return image
    except Exception as e:
        print(f"Image error: {e}")
        return None

def generate_video_fn(prompt):
    if not prompt or prompt.startswith("❌"):
        return None
    try:
        output = video_pipe(
            prompt=prompt,
            negative_prompt=NEGATIVE_PROMPT,
            num_frames=16,
            guidance_scale=2.0,
            num_inference_steps=6,
            generator=torch.Generator(device="cpu").manual_seed(0),
        )
        gif_path = "/content/generated_video.gif"
        export_to_gif(output.frames[0], gif_path)
        torch.cuda.empty_cache()
        return gif_path
    except Exception as e:
        print(f"Video error: {e}")
        return None

def process_all(audio_file):
    text = transcribe_voice(audio_file)
    if text.startswith("❌") or text == "(No speech detected)":
        return text, None, None
    img = generate_image_fn(text)
    vid = generate_video_fn(text)
    return text, img, vid

print("\nLaunching Gradio Web App...")
with gr.Blocks(theme=gr.themes.Soft(primary_hue="indigo"), title="Voice2Vision AI") as demo:
    gr.Markdown("# 🎙️ Voice2Vision AI\n**Speak a scene — watch it come alive as an Image & Video.**")

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 1. Speak or Upload Audio")
            audio_input = gr.Audio(
                sources=["microphone", "upload"],
                type="filepath",
                label="Record or Upload Audio"
            )

            btn_transcribe = gr.Button("📝 1. Transcribe Voice", variant="secondary")
            prompt_box = gr.Textbox(label="Transcribed Prompt (editable)", lines=2)

            with gr.Row():
                btn_image = gr.Button("🖼️ Generate Image", variant="primary")
                btn_video = gr.Button("🎬 Generate Video", variant="primary")

            btn_all = gr.Button("✨ Do All at Once (Transcribe + Image + Video)", variant="stop")

        with gr.Column(scale=1):
            gr.Markdown("### 2. Generated Outputs")
            img_output = gr.Image(label="Generated Image (Dreamlike Diffusion, No Watermark)")
            vid_output = gr.Image(label="Generated Video (AnimateDiff 16-Frame GIF)")

    # Wiring Event Listeners
    btn_transcribe.click(fn=transcribe_voice, inputs=audio_input, outputs=prompt_box)
    btn_image.click(fn=generate_image_fn, inputs=prompt_box, outputs=img_output)
    btn_video.click(fn=generate_video_fn, inputs=prompt_box, outputs=vid_output)
    btn_all.click(fn=process_all, inputs=audio_input, outputs=[prompt_box, img_output, vid_output])

demo.launch(share=True, debug=True)

Unable to import `torchao` Tensor objects. This may affect loading checkpoints serialized with `torchao`


Runtime Device: cuda

[1/3] Loading MOSS-Transcribe-Diarize Speech Model...


Loading weights:   0%|          | 0/683 [00:00<?, ?it/s]

[transformers] `MossTranscribeDiarizeProcessor` defines `feature_extractor_class = 'AutoFeatureExtractor'`, which is deprecated. Register the correct mapping in `AutoFeatureExtractor` instead.



[2/3] Loading Dreamlike Diffusion 1.0 (No watermark)...


...

/tmp/ipykernel_7037/1259788276.py:168: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(primary_hue="indigo"), title="Voice2Vision AI") as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://47776177c251eeb431.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (130 > 77). Running this sequence through the model will result in indexing errors


  0%|          | 0/6 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]